# Breast Cancer Wisconsin — Biomarker Discovery & Explainable AI

**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Source:** UCI Machine Learning Repository  
**Goal:** Train classifiers to distinguish malignant from benign tumors, then use model-explainability techniques (Feature Importance, Permutation Importance, SHAP) to identify the cell-nucleus measurements that matter most.

---

## Project Outline

| # | Phase | Key techniques |
|---|-------|----------------|
| 1 | Exploratory Data Analysis | `.describe()`, correlation heatmap, boxplots |
| 2 | Preprocessing | Label encoding, train/test split, feature scaling |
| 3 | Baseline Model | Logistic Regression + coefficient-based feature ranking |
| 4 | Ensemble Model | Random Forest + Gini-importance feature ranking |
| 5 | Feature Selection | Reduce to top-10 features; re-evaluate |
| 6 | Explainability | SHAP beeswarm plot; cross-method comparison |
| 7 | Conclusions | Biomarker consensus & findings |

---

## Phase 1 — Exploratory Data Analysis

Before touching a model we need to understand the raw data: its shape, feature types, class distribution, data quality, and inter-feature relationships.

### 1.1 Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Reproducibility & plot aesthetics
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

### 1.2 Load & Inspect the Dataset

The CSV contains 569 patient records and 32 columns.  
Each record describes measurements computed from a digitised image of a fine-needle aspirate (FNA) of a breast mass.

In [ ]:
data = pd.read_csv("data.csv")
print("Shape:", data.shape)
data.head()

### 1.3 Column Inventory

In [ ]:
data.info()

**Observations:**
- `id` — unique patient identifier; not a predictive feature.
- `diagnosis` — target variable (`M` = Malignant, `B` = Benign).
- 30 numeric features — three statistical summaries (mean, standard error, worst) for 10 geometric measurements of each cell nucleus.
- `Unnamed: 32` — entirely `NaN`; an artefact of trailing commas in the CSV.

### 1.4 Remove Uninformative Columns

In [ ]:
# Confirm Unnamed: 32 is entirely null
print("Null count:", data['Unnamed: 32'].isnull().sum(), "/ ", len(data))

# Drop artefact column and non-predictive ID
data.drop(columns=['Unnamed: 32', 'id'], inplace=True)
print("Remaining shape:", data.shape)
data.head()

### 1.5 Class Distribution

A significant class imbalance would force us to use stratified splits and pay attention to recall over raw accuracy.

In [ ]:
counts = data['diagnosis'].value_counts()
print(counts)
print(f"\nClass ratio  B : M = {counts['B']} : {counts['M']} ({counts['B']/len(data)*100:.1f}% benign)")

fig, ax = plt.subplots(figsize=(5, 3.5))
counts.plot(kind='bar', ax=ax, color=['steelblue', 'salmon'], edgecolor='white', width=0.5)
ax.set_title("Diagnosis Distribution", fontsize=13, fontweight='bold')
ax.set_xlabel("Diagnosis  (B = Benign, M = Malignant)", fontsize=10)
ax.set_ylabel("Patient Count", fontsize=10)
ax.set_xticklabels(['Benign', 'Malignant'], rotation=0)
for bar in ax.patches:
    ax.annotate(f'{int(bar.get_height())}', (bar.get_x() + bar.get_width()/2, bar.get_height() + 3),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

**Finding:** The dataset is moderately imbalanced (≈ 63% benign, 37% malignant). Stratified splitting will preserve this ratio across train and test sets.

### 1.6 Data Quality Check

In [ ]:
print("Missing values per column:")
print(data.isnull().sum().sum(), "total missing values")
print()
print("Duplicate rows:", data.duplicated().sum())

No missing values and no duplicate rows — the dataset is clean.

### 1.7 Descriptive Statistics

In [ ]:
data.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

**Observation:** Features span vastly different numeric ranges (e.g. `area_mean` is in the thousands while `smoothness_mean` is < 0.3). Standardisation will be essential before training distance/gradient-sensitive models.

### 1.8 Correlation Heatmap

**Hypothesis:** Size-related features (radius, perimeter, area) should be strongly correlated because they all quantify geometric extent.

In [ ]:
corr = data.drop('diagnosis', axis=1).corr()

plt.figure(figsize=(18, 15))
mask = np.triu(np.ones_like(corr, dtype=bool))          # show only lower triangle
sns.heatmap(
    corr, mask=mask, cmap='coolwarm', center=0,
    annot=False, linewidths=0.4,
    vmin=-1, vmax=1
)
plt.title("Feature Correlation Matrix", fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

**Findings:**
- **Strong positive clusters** among `radius`, `perimeter`, and `area` (mean, SE, and worst variants) — expected geometrically.
- **Mean ↔ worst** pairs are also highly correlated for most features.
- This multicollinearity means several features carry redundant information. Feature selection (Phase 5) will exploit this.

### 1.9 Separability Check — Boxplot by Diagnosis

A quick visual to confirm that at least some features cleanly separate the two classes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, feat in zip(axes, ['area_mean', 'concavity_mean', 'texture_mean']):
    sns.boxplot(x='diagnosis', y=feat, data=data, ax=ax,
                palette={'B': 'steelblue', 'M': 'salmon'},
                width=0.5, fliersize=3)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel("Diagnosis", fontsize=9)
    ax.set_xticklabels(['Benign', 'Malignant'])

fig.suptitle("Feature Distributions by Diagnosis Class", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Finding:** `area_mean` and `concavity_mean` show near-complete class separation, making them strong candidate features. `texture_mean` has more overlap — it will be less important in ensemble models.

---

## Phase 2 — Preprocessing

### 2.1 Encode the Target Variable

Scikit-learn classifiers require a numeric target. We use `LabelEncoder` which maps `B → 0` and `M → 1`.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['diagnosis'] = le.fit_transform(data['diagnosis'])

print("Encoding map:", dict(zip(le.classes_, le.transform(le.classes_))))
print(data['diagnosis'].value_counts())

### 2.2 Feature / Target Split

In [ ]:
X = data.drop('diagnosis', axis=1)
y = data['diagnosis']

print("Features shape:", X.shape)
print("Target shape:  ", y.shape)

### 2.3 Stratified Train / Test Split

`stratify=y` ensures the 63 / 37 class ratio is preserved in both splits.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape, "| Class balance:")
print(y_train.value_counts(normalize=True).round(3))
print()
print("Test size: ", X_test.shape, "| Class balance:")
print(y_test.value_counts(normalize=True).round(3))

### 2.4 Feature Scaling

Logistic Regression optimises a loss function using gradient descent; features on larger scales dominate and slow convergence. `StandardScaler` transforms each feature to zero mean and unit variance.

> **Important:** We `fit` on training data only, then `transform` both sets. Fitting on the full dataset would leak test-set statistics into training — a form of data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled  = scaler.transform(X_test)        # transform only

print("Train mean (first 3 features):", X_train_scaled[:, :3].mean(axis=0).round(4))
print("Train std  (first 3 features):", X_train_scaled[:, :3].std(axis=0).round(4))

---

## Phase 3 — Baseline Model: Logistic Regression

Logistic Regression is the natural baseline for binary classification. Its learned **coefficients** provide a first look at feature importance: large absolute coefficient → large influence on the log-odds of malignancy.

### 3.1 Train

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42, max_iter=10000)
lr.fit(X_train_scaled, y_train)

### 3.2 Evaluate

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred_lr = lr.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}\n")
print(classification_report(y_test, y_pred_lr, target_names=['Benign', 'Malignant']))

fig, ax = plt.subplots(figsize=(4, 3.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr),
                       display_labels=['Benign', 'Malignant']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title("Logistic Regression — Confusion Matrix", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Feature Ranking via Coefficients

The magnitude of each coefficient reflects how strongly that feature shifts the model's output toward malignant (positive) or benign (negative).

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

# ── Plot top 15 ──
top15_coef = coef_df.head(15)
colors = ['salmon' if c > 0 else 'steelblue' for c in top15_coef['Coefficient']]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top15_coef['Feature'][::-1], top15_coef['Coefficient'][::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title("Logistic Regression — Top 15 Feature Coefficients", fontsize=12, fontweight='bold')
ax.set_xlabel("Coefficient Value  (positive → malignant)")
plt.tight_layout()
plt.show()

print(coef_df.head(10).to_string(index=False))

**Findings:**
- `texture_worst`, `radius_se`, and `concavity_worst` top the logistic regression ranking.
- Positive coefficients push the model toward malignant; negative coefficients toward benign.
- Logistic Regression is a **linear** model — it can only capture linear relationships between features and the log-odds of malignancy.

---

## Phase 4 — Ensemble Model: Random Forest

Random Forest builds many decision trees on bootstrapped subsets of the data and aggregates their predictions. Its built-in **Gini importance** measures how much each feature reduces impurity across all trees — capturing non-linear relationships that Logistic Regression cannot.

> Note: Random Forest does not require feature scaling.

### 4.1 Train

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

### 4.2 Evaluate

In [ ]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}\n")
print(classification_report(y_test, y_pred_rf, target_names=['Benign', 'Malignant']))

fig, ax = plt.subplots(figsize=(4, 3.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_rf),
                       display_labels=['Benign', 'Malignant']).plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title("Random Forest — Confusion Matrix", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Feature Ranking via Gini Importance

In [ ]:
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

top15_imp = feat_imp.head(15)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top15_imp['Feature'][::-1], top15_imp['Importance'][::-1],
        color='mediumseagreen', edgecolor='white')
ax.set_title("Random Forest — Top 15 Feature Importances (Gini)", fontsize=12, fontweight='bold')
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()

print(feat_imp.head(10).to_string(index=False))

**Findings:**
- `perimeter_worst`, `area_worst`, and `radius_worst` dominate — all size-related features.
- This contrasts with Logistic Regression where `texture_worst` ranked first, illustrating that different model classes can legitimately disagree on feature rankings.
- Random Forest generalises better here thanks to its ability to capture feature interactions.

---

## Phase 5 — Feature Selection: Simplify with the Top 10 Features

**Motivation:** We observed heavy multicollinearity (Phase 1) and a consistent top-10 cluster of important features (Phase 4). A leaner model with only the most informative features is:
- Easier to interpret
- Faster to train and deploy
- Less susceptible to overfitting

We retain the top-10 Random Forest features and retrain Logistic Regression to test whether accuracy holds.

### 5.1 Select Top-10 Features

In [ ]:
top10_features = feat_imp.head(10)['Feature'].tolist()
print("Top-10 selected features:")
for i, f in enumerate(top10_features, 1):
    print(f"  {i:2d}. {f}")

X_top10 = X[top10_features]
print(f"\nReduced feature matrix shape: {X_top10.shape}")

### 5.2 Retrain / Re-evaluate on Top-10

In [ ]:
X_train_10, X_test_10, y_train_10, y_test_10 = train_test_split(
    X_top10, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler_10 = StandardScaler()
X_train_10_scaled = scaler_10.fit_transform(X_train_10)
X_test_10_scaled  = scaler_10.transform(X_test_10)

lr_top10 = LogisticRegression(random_state=42, max_iter=10000)
lr_top10.fit(X_train_10_scaled, y_train_10)

y_pred_top10 = lr_top10.predict(X_test_10_scaled)

print(f"Accuracy (top-10 features): {accuracy_score(y_test_10, y_pred_top10):.4f}")
print(classification_report(y_test_10, y_pred_top10, target_names=['Benign', 'Malignant']))

**Finding:** Logistic Regression on the top-10 features matches (or closely approaches) the full-feature accuracy, confirming that the remaining 20 features were largely redundant. A 67% reduction in dimensionality with no meaningful loss in performance.

---

## Phase 6 — Explainability: SHAP (SHapley Additive exPlanations)

Feature importance scores (Phase 4) answer *which* features the model used most on average. SHAP answers a richer question: **how much did each feature contribute to a specific prediction, and in which direction?**

SHAP is rooted in cooperative game theory (Shapley values). For each prediction, it computes the marginal contribution of every feature across all possible feature orderings — guaranteeing a theoretically sound, model-agnostic attribution.

### 6.1 Build the SHAP Explainer

In [ ]:
import shap
shap.initjs()

# TreeExplainer is the exact (non-approximate) method for tree-based models
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)   # shape: (n_samples, n_features, n_classes)

print("SHAP values type: ", type(shap_values))
print("SHAP values shape:", shap_values.shape)
print("  → (samples, features, classes)")
print()
print("We slice [:, :, 1] to extract class-1 (Malignant) SHAP values")

### 6.2 Beeswarm Summary Plot

Each dot is one patient. Dot colour = feature value (red = high, blue = low). X-axis position = SHAP value = contribution to the model's malignancy score.

- **Right of zero** → increases predicted probability of malignancy  
- **Left of zero** → decreases predicted probability of malignancy

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values[:, :, 1],
    X_test,
    feature_names=X.columns,
    plot_type="dot",
    show=False
)
plt.title("SHAP Beeswarm Plot — Feature Contributions (Malignant class)",
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 6.3 Interpreting the Plot

| Feature | Pattern | Medical meaning |
|---------|---------|----------------|
| `perimeter_worst` | Red dots far right | Large worst-case perimeter → strong push toward malignant |
| `area_worst` | Red dots far right | Same story; area and perimeter are geometrically linked |
| `concave points_worst` | Red right, blue left | Irregular boundary → malignant; smooth boundary → benign |
| `radius_worst` | Consistent with above | Tumour size is the dominant discriminator |
| `texture_mean` | Spread, mixed colours | Moderate but more variable contribution |

---

## Phase 7 — Conclusions

### Cross-Method Feature Importance Comparison

| Rank | Logistic Regression (coefficient) | Random Forest (Gini) | SHAP |
|------|----------------------------------|----------------------|------|
| 1 | texture_worst | perimeter_worst | perimeter_worst |
| 2 | radius_se | area_worst | area_worst |
| 3 | concavity_worst | radius_worst | concave points_worst |
| 4 | concave points_worst | concave points_worst | radius_worst |
| 5 | area_worst | concavity_mean | concavity_worst |

**Three methods, one consistent story:** tumour size (`radius`, `perimeter`, `area`) and boundary irregularity (`concavity`, `concave points`) — specifically their *worst* measurements — are the strongest predictors of malignancy. This aligns with established medical knowledge.

### Key Takeaways

1. **SHAP and Random Forest importance converge** on the same top features, giving us high confidence in the identified biomarkers.
2. **Logistic Regression disagrees partially** (e.g. `texture_worst` ranks first) because it captures only linear effects, while RF and SHAP capture non-linear interactions.
3. **Feature reduction works:** dropping from 30 to 10 features caused negligible accuracy loss, confirming heavy redundancy from correlated size measurements.
4. **SHAP adds directionality:** unlike plain importance scores, SHAP reveals *that* high perimeter and area values increase malignancy probability — a clinically interpretable finding.

### Model Performance Summary

| Model | Features | Accuracy |
|-------|----------|----------|
| Logistic Regression | All 30 | ~96% |
| Random Forest | All 30 | ~97% |
| Logistic Regression | Top 10 only | ~96% |

---

*Built with Scikit-Learn, SHAP, Pandas, Matplotlib, and Seaborn.*